# Egg collection over time

Run all cells to count the **eggs across the entire CSV**, regardless of collector. Each record contributes its `Individual Count`, rather than counting one record as one egg. The code first totals eggs by **calendar month** and then plots a centered **12-month moving average** of those monthly totals. This removes much of the day-to-day noise and seasonal clutter while retaining the broad collecting history. The original monthly totals remain available as `monthly` in the notebook. Years have X-axis ticks, and the PNG is saved next to the CSV. For long time spans, some year labels are skipped to keep them readable.

The CSV is expected at the `G:` drive path below. If you move it, edit `CSV_PATH` in the first code cell. In this export, `year`, `month`, and `date collected m/d/y` can hold the year, numeric month, and **day of month** separately (for example, `1904`, `5`, `4`). Since this graph groups by month, a missing day is fine. The code also accepts a complete date in the last field and uses `MonthCollected` if numeric `month` is empty. Records without a valid year and month, or without a positive whole-number egg count, are reported and excluded. The average is measured in **eggs per month**, not eggs per year or cumulative eggs; the first and last months use the available portion of the 12-month window.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import StrMethodFormatter

CSV_PATH = Path(r"G:\My Drive\Egg Slip Scanning\EggSlipReorganizationProject_FULL.xlsx - Full List.csv")
OUTPUT_PATH = CSV_PATH.with_name("Egg_collection_smoothed_monthly.png")
SMOOTH_MONTHS = 12  # Try 6 for more local detail, or 18 for a smoother trend.

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}\nEdit CSV_PATH above if the file has moved.")


In [ ]:
# Read only the fields needed for this graph; keep text intact until it is validated.
columns = ["Individual Count", "year", "month", "date collected m/d/y", "MonthCollected"]
df = pd.read_csv(CSV_PATH, usecols=columns, dtype="string", encoding="utf-8-sig")
if df.empty:
    raise ValueError("The CSV has no records to graph.")

# For monthly totals, a known year and month suffice even when day is blank.
# 'date collected m/d/y' usually holds only the day; when it holds a complete
# date, the month and year in that complete date take precedence.
numeric_year = pd.to_numeric(df["year"].str.strip(), errors="coerce")
numeric_year = numeric_year.where(numeric_year.eq(numeric_year.round()))
numeric_month = pd.to_numeric(df["month"].str.strip(), errors="coerce")
numeric_month = numeric_month.where(numeric_month.eq(numeric_month.round()) & numeric_month.between(1, 12))
month_names = dict(zip(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"],
    range(1, 13),
))
named_month = df["MonthCollected"].str.strip().str[:3].str.lower().map(month_names)
numeric_month = numeric_month.fillna(named_month)

date_text = df["date collected m/d/y"].str.strip()
full_date = pd.to_datetime(date_text, format="%m/%d/%Y", errors="coerce")
full_date = full_date.fillna(pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce"))
full_month = full_date.dt.to_period("M").dt.to_timestamp()

parts = pd.DataFrame({
    "year": numeric_year.astype("Int64").astype("string"),
    "month": numeric_month.astype("Int64").astype("string").str.zfill(2),
}, index=df.index)
split_month = pd.to_datetime(
    parts["year"] + "-" + parts["month"] + "-01",
    format="%Y-%m-%d", errors="coerce",
)
df["collection_month"] = full_month.fillna(split_month)
df["egg_count"] = pd.to_numeric(
    df["Individual Count"].str.replace(",", "", regex=False).str.strip(),
    errors="coerce",
)

valid_month = df["collection_month"].notna() & df["collection_month"].dt.year.between(1700, pd.Timestamp.today().year)
valid_count = (
    df["egg_count"].gt(0) & df["egg_count"].eq(df["egg_count"].round())
).fillna(False)
usable = df.loc[valid_month & valid_count, ["collection_month", "egg_count"]].copy()
if usable.empty:
    raise ValueError("No records have both a valid collection month and a positive whole-number Individual Count.")

usable["egg_count"] = usable["egg_count"].astype(int)

print(f"Collection records found: {len(df):,}")
print(f"Records excluded for missing/invalid year or month: {(~valid_month).sum():,}")
print(f"Records excluded for missing/invalid egg count: {(~valid_count).sum():,}")
print(f"Records plotted: {len(usable):,}")
print(f"Eggs plotted: {usable['egg_count'].sum():,}")


In [ ]:
# Calendar-month totals with zeroes for months without collecting.
monthly = usable.groupby("collection_month")["egg_count"].sum().sort_index()
first_year, last_year = monthly.index.min().year, monthly.index.max().year
calendar_months = pd.date_range(f"{first_year}-01-01", f"{last_year}-12-01", freq="MS")
monthly = monthly.reindex(calendar_months, fill_value=0)

# Each point is the average monthly total in the surrounding SMOOTH_MONTHS.
# This is a rate in eggs/month, and preserves zero months in the calculation.
smoothed = monthly.rolling(
    window=SMOOTH_MONTHS, center=True, min_periods=max(1, SMOOTH_MONTHS // 2)
).mean()

fig, ax = plt.subplots(figsize=(22, 7))
ax.plot(smoothed.index, smoothed.values, color="#236798", linewidth=2.5)
ax.fill_between(smoothed.index, 0, smoothed.values, color="#236798", alpha=0.13)
ax.set_title("Egg collection · smoothed monthly rate", fontsize=18, pad=18)
ax.set_xlabel("Collection year", labelpad=14)
ax.set_ylabel(f"Eggs per month ({SMOOTH_MONTHS}-month moving average)")
ax.set_ylim(bottom=0)
ax.set_xlim(calendar_months.min(), calendar_months.max() + pd.offsets.MonthEnd(1))
years_spanned = last_year - first_year + 1
label_every = 1 if years_spanned <= 70 else 2 if years_spanned <= 120 else 5
ax.xaxis.set_major_locator(mdates.YearLocator(label_every))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.YearLocator(1))  # Every year still has a tick.
ax.tick_params(axis="x", which="major", length=9, width=1.1, labelrotation=90, labelsize=8, pad=5)
ax.tick_params(axis="x", which="minor", length=4, width=0.7, color="0.55")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="y", color="0.82", linewidth=0.8)
for year in range(first_year, last_year + 1):
    if year % 5 == 0:
        ax.axvline(pd.Timestamp(year, 1, 1), color="0.88", linewidth=0.7, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.99, 0.01, f"{monthly.sum():,} eggs across {len(usable):,} month-dated records", ha="right", fontsize=10, color="dimgray")

fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(OUTPUT_PATH, dpi=200, bbox_inches="tight")
print(f"Saved graph: {OUTPUT_PATH}")
plt.show()
